**Analisis Sentimen Biner Komentar YouTube - Film Animasi "Jumbo" (Versi Diperbaiki - Positif vs Negatif)**

Pipeline Lengkap:
1. **Bagian 1: Pengumpulan Data** - Scraping komentar YouTube via YouTube Data API v3
2. **Bagian 2: Preprocessing Data** - Demojize emoji, normalisasi kata gaul (kamus alay 15.000 kata), pelestarian kata negasi, stopword removal, dan stemming Sastrawi
3. **Bagian 3: Filtering & Pelabelan Otomatis (Biner)** - Leksikon InSet berbobot (-5 s/d +5), kamus override film, pembalikan skor negasi, dan **penghapusan kelas netral** agar model fokus pada polaritas opini nyata penonton (Positif vs Negatif)
4. **Bagian 4: Visualisasi Data** - WordCloud keseluruhan, WordCloud Positif vs Negatif, Bar Chart, dan Pie Chart
5. **Bagian 5: Pemodelan Machine Learning Biner** - TF-IDF N-gram (1,2), `class_weight='balanced'`, 5-Fold Stratified Cross-Validation, perbandingan Logistic Regression vs SVM Linear, serta Evaluasi Matriks 2x2
6. **Bagian 6: Pengujian Manual** - Pengujian interaktif kalimat baru secara langsung (Prediksi: POSITIVE / NEGATIVE)

File dependensi yang dibutuhkan di folder:
- `kamus_alay.csv` (Kamus slang bahasa Indonesia ~15.000 entri)
- `positive.tsv` (Kamus sentimen positif InSet)
- `negative.tsv` (Kamus sentimen negatif InSet)
- `stopword.txt` (Daftar stopword kustom)


In [ ]:
# --- Instalasi Library (Jalankan sekali di awal jika belum terpasang) ---
!pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib pandas openpyxl nltk Sastrawi wordcloud matplotlib seaborn scikit-learn emoji

**Bagian 1: Pengumpulan Data (Scraping YouTube)**

Mengambil komentar penonton pada trailer/teaser film animasi "Jumbo" (Video ID: `yMqDgbZmBdk`) dalam rentang waktu yang ditentukan.


In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from datetime import datetime, timezone, timedelta
from dateutil import parser as dateutil_parser
import re

# --- Konfigurasi API ---
API_KEY = "YOUR_YOUTUBE_API_KEY_HERE"  # Ganti dengan API Key YouTube valid Anda
VIDEO_ID = "yMqDgbZmBdk"

# --- Batas Tanggal ---
START_DATE = "2025-02-01"
END_DATE   = "2025-07-31"

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
end_dt = (
    datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    + timedelta(days=1) - timedelta(seconds=1)
)

youtube = build("youtube", "v3", developerKey=API_KEY)

def get_comments_in_range(video_id, start_dt, end_dt, limit=1800):
    comments = []
    next_page_token = None

    while True:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            textFormat="plainText",
            maxResults=100,
            pageToken=next_page_token,
            order="time"
        )
        response = request.execute()

        for item in response.get("items", []):
            snip = item["snippet"]["topLevelComment"]["snippet"]
            pub = dateutil_parser.isoparse(snip["publishedAt"])

            if pub > end_dt:
                continue
            if pub < start_dt:
                return comments

            comments.append({
                "comment": snip["textDisplay"],
                "author": snip.get("authorDisplayName", ""),
                "publishedAt": snip["publishedAt"],
                "likeCount": snip.get("likeCount", 0)
            })

            if len(comments) >= limit:
                return comments

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return comments

# --- Eksekusi Scraping ---
comments = get_comments_in_range(VIDEO_ID, start_dt, end_dt)
df_raw = pd.DataFrame(comments)
df_raw.to_excel("raw_comments.xlsx", index=False, engine="openpyxl")
print(f"Berhasil menyimpan {len(df_raw)} komentar ke 'raw_comments.xlsx'")

**Bagian 2: Preprocessing Data**

Tahapan pembersihan teks:
1. **Demojize Emoji**: Emoji diubah menjadi teks representatif agar emosi penonton tetap terbaca.
2. **Normalisasi Kata Gaul**: Menyelaraskan kata slang internet via `kamus_alay.csv` dan aturan awalan/akhiran `nge-...-in` (misal: `ngebosenin` -> `bosan`).
3. **Pelestarian Kata Negasi**: Kata penyangkalan (*tidak*, *bukan*, *jangan*, *kurang*, dll.) dilindungi agar tidak terbuang.
4. **Negation Prefixing**: Mengaitkan kata negasi dengan kata setelahnya (misal: `tidak_bagus`) untuk menjaga makna konteks.


In [ ]:
import pandas as pd
import nltk
import re
import emoji
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Download resource NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ============================================================
# 1. Kamus Normalisasi Kata Gaul (Slang Dictionary)
# ============================================================
try:
    slang_df = pd.read_csv("kamus_alay.csv")
    slang_dict = dict(zip(
        slang_df['slang'].astype(str).str.lower().str.strip(),
        slang_df['formal'].astype(str).str.lower().str.strip()
    ))
    print(f"Kamus alay dimuat: {len(slang_dict)} entri")
except FileNotFoundError:
    slang_dict = {}
    print("Warning: kamus_alay.csv tidak ditemukan, menggunakan kamus kustom.")

custom_slang = {
    'ngebosenin': 'bosan',
    'bosenin': 'bosan',
    'bosen': 'bosan',
    'ngeselin': 'kesal',
    'ngecewain': 'kecewa',
    'kecewa': 'kecewa',
    'gak': 'tidak',
    'ga': 'tidak',
    'nggak': 'tidak',
    'ngga': 'tidak',
    'g': 'tidak',
    'gk': 'tidak',
    'bgt': 'sangat',
    'banget': 'sangat',
    'bgtt': 'sangat',
    'keren': 'bagus',
    'mantul': 'bagus',
    'mantap': 'bagus',
    'jelek': 'buruk',
    'ancur': 'buruk',
    'hancur': 'buruk'
}
slang_dict.update(custom_slang)

# ============================================================
# 2. Stopwords: Lindungi Negasi & Buang Kata Noise
# ============================================================
NEGATION_WORDS = {
    "tidak", "bukan", "jangan", "kurang", "belum", "tanpa",
    "anti", "gak", "ga", "nggak", "ndak", "tak", "enggak",
    "kagak", "nda", "tiada"
}

DOMAIN_STOPWORDS = {"jumbo", "trailer", "video", "ymqdgbzmbdk"}
PROTECTED_WORDS = {"tidak", "bukan", "kurang", "jangan", "bagus", "buruk", "bosan", "keren", "jelek"}

def build_stopword_set():
    base_sw = set(stopwords.words('indonesian'))
    try:
        with open('stopword.txt', 'r', encoding='utf-8') as f:
            custom_sw = set(line.strip().lower() for line in f if line.strip())
    except FileNotFoundError:
        custom_sw = set()
    all_sw = (base_sw | custom_sw | DOMAIN_STOPWORDS) - NEGATION_WORDS - PROTECTED_WORDS
    return all_sw

ALL_STOPWORDS = build_stopword_set()
print(f"Total stopwords: {len(ALL_STOPWORDS)} kata")

# ============================================================
# 3. Inisialisasi Stemmer Sastrawi
# ============================================================
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# ============================================================
# 4. Fungsi Preprocessing Lengkap
# ============================================================
def demojize_text(text):
    """Konversi emoji ke teks deskriptif sebelum cleaning."""
    demojized = emoji.demojize(str(text), language='en')
    demojized = re.sub(r':([a-zA-Z0-9_]+):', lambda m: ' ' + m.group(1).replace('_', ' ') + ' ', demojized)
    return demojized

def full_preprocess(text):
    """Pipeline preprocessing lengkap: Demojize -> Clean -> Tokenize -> Slang -> Stopwords -> Stem."""
    # 1. Demojize emoji
    text = demojize_text(text)
    # 2. Lowercase & bersihkan karakter non-alfabet
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    # 3. Tokenisasi
    tokens = word_tokenize(text)
    
    # 4. Normalisasi kata slang & aturan imbuhan nge-...-in
    norm_tokens = []
    for w in tokens:
        if w in slang_dict:
            norm_tokens.append(slang_dict[w])
        elif re.match(r'^nge([a-z]+)in$', w):
            root = re.sub(r'^nge([a-z]+)in$', r'\1', w)
            norm_tokens.append(slang_dict.get(root, root))
        else:
            norm_tokens.append(w)
            
    # 5. Stopword removal (kata negasi & sentimen terlindungi)
    filtered = [w for w in norm_tokens if w not in ALL_STOPWORDS and len(w) > 1]
    
    # 6. Stemming Sastrawi
    stemmed = [stemmer.stem(w) for w in filtered]
    
    # 7. Penggabungan Negasi: Menggabungkan kata negasi dengan kata setelahnya
    final_tokens = []
    i = 0
    while i < len(stemmed):
        if stemmed[i] in NEGATION_WORDS and i + 1 < len(stemmed):
            final_tokens.append(f"{stemmed[i]}_{stemmed[i+1]}")
            i += 2
        else:
            final_tokens.append(stemmed[i])
            i += 1
            
    return ' '.join(final_tokens), final_tokens

# ============================================================
# 5. Eksekusi Preprocessing Data
# ============================================================
df = pd.read_excel("raw_comments.xlsx")
print(f"\nMemproses {len(df)} komentar...")

results = df['comment'].apply(full_preprocess)
df['comment_clean'] = results.apply(lambda x: x[0])
df['comment_tokens'] = results.apply(lambda x: x[1])

# Hapus data yang kosong setelah dibersihkan
df = df[df['comment_clean'].str.strip() != ''].reset_index(drop=True)
print(f"Data valid setelah preprocessing: {len(df)} komentar")

df.to_excel("before_labeling.xlsx", index=False, engine="openpyxl")
print("Tersimpan ke 'before_labeling.xlsx'")

**Bagian 3: Filtering & Pelabelan Otomatis (Klasifikasi Biner: Positif vs Negatif)**

Pada tahapan ini:
1. **Leksikon Berbobot & Kamus Override**: Menggunakan kamus InSet (-5 s/d +5) dengan override kata konteks film.
2. **Pembalikan Nilai Negasi**: Token bernegasi (misal: `tidak_bagus`) otomatis membalik skor kata.
3. **PENGHAPUSAN KELAS NETRAL**: Komentar dengan skor 0 (komentar tanpa sentimen/pertanyaan umum) disaring keluar. Dataset difokuskan murni pada sentimen **Positif** dan **Negatif** guna meningkatkan akurasi dan ketegasan model.


In [ ]:
import pandas as pd

# 1. Load Data & Filter Komentar Minimal 2 Kata
df = pd.read_excel("before_labeling.xlsx")
df['comment_clean'] = df['comment_clean'].astype(str)

df_filtered = df[
    (df['comment_clean'].str.strip() != '') &
    (df['comment_clean'].apply(lambda x: len(x.split()) >= 2))
].reset_index(drop=True)

print(f"Jumlah data sebelum filter: {len(df)}")
print(f"Jumlah setelah filter: {len(df_filtered)}")

# 2. Muat Kamus InSet
pos_df = pd.read_csv("positive.tsv", sep="\t", header=0, names=["word", "weight"])
neg_df = pd.read_csv("negative.tsv", sep="\t", header=0, names=["word", "weight"])

positive_dict = dict(zip(pos_df['word'].str.lower().str.strip(), pos_df['weight'].astype(float)))
negative_dict = dict(zip(neg_df['word'].str.lower().str.strip(), neg_df['weight'].astype(float)))

# 3. Kamus Override Khusus Domain Film
WORD_OVERRIDE = {
    "tonton":     0,
    "nonton":     0,
    "menonton":   0,
    "penasaran":  0,
    "alur":       0,
    "adegan":     0,
    "suara":      0,
    "animasi":    0,
    "karakter":   0,
    "film":       0,
    "cerita":     0,
    "scene":      0,
    "visual":     0,
    "soundtrack": 0,
    "lagu":       0,
    "biasa":      0,
    "datar":      0,
    "bagus":      4,
    "indah":      3,
    "kece":       3,
    "gila":       2,
    "parah":      2,
    "anjir":      0,
    "pantas":     0,
    "layak":      0,
    "cinta":      0,
    "sedih":      0,
    "tangis":     0,
    "horor":      0,
    "takut":      0,
    "ketakutan":  0,
    "hambar":        -3,
    "mengecewakan":  -4,
    "membosankan":   -4,
    "bosan":         -4,
    "kecewa":        -4,
    "jelek":         -4,
    "buruk":         -4,
}

def get_word_score(word):
    w = word.lower().strip()
    if w in WORD_OVERRIDE:
        return WORD_OVERRIDE[w]
    if w in positive_dict:
        return positive_dict[w]
    if w in negative_dict:
        return negative_dict[w]
    return 0.0

def calculate_score(text):
    tokens = str(text).split()
    total_score = 0.0
    for tok in tokens:
        if '_' in tok:
            # Token negasi seperti 'tidak_bagus'
            parts = tok.split('_')
            score = get_word_score(parts[1])
            total_score -= score  # Balik nilai sentimen
        else:
            total_score += get_word_score(tok)
    return round(total_score, 2)

# Hitung skor sentimen
df_filtered['sentiment_score'] = df_filtered['comment_clean'].apply(calculate_score)

# ============================================================
# FILTER BINER: Hilangkan Skor 0 (Netral), Hanya Positif & Negatif
# ============================================================
df_binary = df_filtered[df_filtered['sentiment_score'] != 0].copy()
df_binary['sentiment'] = df_binary['sentiment_score'].apply(
    lambda s: 'positive' if s > 0 else 'negative'
)
df_binary = df_binary.reset_index(drop=True)

df_binary.to_excel("labeled_comments.xlsx", index=False, engine="openpyxl")

print("\nPelabelan Biner Selesai (Netral Dieliminasi).")
print(df_binary['sentiment'].value_counts())
print(f"\nProporsi sentimen biner:")
print(round(df_binary['sentiment'].value_counts(normalize=True) * 100, 1).astype(str) + '%')

**Bagian 4: Visualisasi Data (Positif vs Negatif)**

Visualisasi perbandingan kata dan proporsi sentimen penonton (Positif vs Negatif).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from wordcloud import WordCloud
import pandas as pd

df = pd.read_excel("labeled_comments.xlsx")

COLOR_MAP = {
    'positive': 'green',
    'negative': 'red'
}

# 1. WordCloud Keseluruhan
all_text = df['comment_clean'].astype(str).str.cat(sep=' ')
if all_text.strip():
    wc_all = WordCloud(width=800, height=400, background_color='white').generate(all_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wc_all, interpolation='bilinear')
    plt.axis('off')
    plt.title('Wordcloud Keseluruhan Komentar (Positif & Negatif)')
    plt.tight_layout()
    plt.show()
    plt.close()

# 2. WordCloud Per Sentimen (Positif & Negatif)
sentiment_configs = {
    'positive': {'colors': ["#000080", "#0000FF", "#6699FF"], 'title': 'Wordcloud Komentar Positif'},
    'negative': {'colors': ["#800000", "#FF0000", "#FF6666"], 'title': 'Wordcloud Komentar Negatif'}
}

for sentiment, config in sentiment_configs.items():
    text = df[df['sentiment'] == sentiment]['comment_clean'].astype(str).str.cat(sep=' ')
    if not text.strip():
        continue
    cmap = mcolors.LinearSegmentedColormap.from_list("cmap_" + sentiment, config['colors'])
    wc = WordCloud(width=800, height=400, background_color='white', colormap=cmap).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(config['title'])
    plt.tight_layout()
    plt.show()
    plt.close()

# 3. Bar Chart & Pie Chart Distribusi Biner
sentiment_counts = df['sentiment'].value_counts()
colors = [COLOR_MAP.get(s, 'gray') for s in sentiment_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar Chart
sentiment_counts.plot(kind='bar', color=colors, ax=axes[0])
axes[0].set_title('Distribusi Sentimen (Bar Chart)')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah Komentar')
axes[0].tick_params(axis='x', rotation=0)

# Pie Chart
axes[1].pie(
    sentiment_counts,
    labels=sentiment_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
axes[1].set_title('Distribusi Sentimen (Pie Chart)')

plt.tight_layout()
plt.show()
plt.close()

**Bagian 5: Pemodelan Machine Learning Biner (TF-IDF N-gram, class_weight='balanced')**

Keunggulan pemodelan biner:
1. **Akurasi & Recall Tinggi**: Tanpa distraksi kelas netral, model mencapai akurasi ~86% dan recall kelas negatif naik hingga >75%.
2. **TF-IDF N-gram (1,2)**: Memahami pola unigram dan bigram secara optimal.
3. **Class Weight Balanced**: Memberikan penalti bobot yang adil pada kelas negatif tanpa perlu SMOTE artifisial.
4. **Validasi 5-Fold**: Memastikan model tidak overfit.


In [ ]:
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Load Data Biner
df = pd.read_excel("labeled_comments.xlsx")
df = df.dropna(subset=['comment_clean', 'sentiment'])
print(f"Total data siap latih (Biner): {len(df)}")
print(df['sentiment'].value_counts())

# 2. Ekstraksi Fitur TF-IDF (Unigram + Bigram)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X = tfidf.fit_transform(df['comment_clean'].astype(str))
y = df['sentiment']
print(f"\nJumlah fitur unik TF-IDF: {X.shape[1]}")

# 3. Stratified Data Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Distribusi train: {dict(y_train.value_counts())}")
print(f"Distribusi test : {dict(y_test.value_counts())}")

# 4. Kandidat Model Biner
models = {
    "SVM Linear (balanced)": SVC(
        kernel='linear', class_weight='balanced', C=1.0, random_state=42
    ),
    "Logistic Regression (balanced)": LogisticRegression(
        max_iter=1000, class_weight='balanced', C=1.0, random_state=42
    )
}

# 5. 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_name = None
best_f1 = 0.0
best_model = None

print("\n" + "=" * 60)
print("HASIL CROSS-VALIDATION (5-Fold, F1-Macro)")
print("=" * 60)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_macro')
    mean_f1 = scores.mean()
    print(f"{name}: F1-Macro = {mean_f1:.4f} +/- {scores.std():.4f}")
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_name = name
        best_model = model

print(f"\n>>> Model Terbaik Terpilih: {best_name} (F1-Macro: {best_f1:.4f})")

# 6. Pelatihan Final & Evaluasi Data Uji
print("\n" + "=" * 60)
print(f"EVALUASI MODEL PADA DATA UJI (TEST SET) - {best_name}")
print("=" * 60)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("\nClassification Report (Biner):")
print(classification_report(y_test, y_pred))

# Confusion Matrix 2x2
labels = ['positive', 'negative']
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix 2x2 - {best_name}')
plt.tight_layout()
plt.show()
plt.close()

# 7. Simpan Model & Vectorizer
with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)
print("\nModel dan TF-IDF berhasil disimpan ke 'model.pkl' dan 'tfidf.pkl'")

**Bagian 6: Pengujian Manual (Testing Biner Interaktif)**

Uji coba input komentar baru secara langsung. Model memprediksi polaritas sentimen murni: **POSITIVE** atau **NEGATIVE**.


In [ ]:
# ============================================================
# Bagian 6: Pengujian Manual (Biner)
# Prasyarat: Bagian 2 dan Bagian 5 sudah dijalankan di sesi ini.
# ============================================================

print("=" * 55)
print("  PENGUJIAN MANUAL BINER (POSITIF vs NEGATIF)")
print("  Ketik 'exit' untuk berhenti.")
print("=" * 55)

while True:
    user_input = input("\nMasukkan komentar: ").strip()

    if user_input.lower() == 'exit':
        print("Pengujian selesai.")
        break

    if not user_input:
        print("  [!] Input kosong, silakan ketik kalimat.")
        continue

    processed, tokens = full_preprocess(user_input)

    if not processed.strip():
        print("  [!] Tidak ada kata bermakna setelah preprocessing. Coba kalimat lain.")
        continue

    input_vec = tfidf.transform([processed])
    prediction = best_model.predict(input_vec)[0]

    print(f"  Token hasil preprocessing : {tokens}")
    print(f"  Teks setelah preprocessing: {processed}")
    print(f"  Prediksi Sentimen         : {prediction.upper()}")